In [1]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(
    name="my_documents"
)

collection.add(
    documents=[
        "Pakistan won the cricket match.",
        "Babar scored a century in cricket.",
        "Apple stock price increased today.",
        "Microsoft invested heavily in AI."
    ],
    ids=["1", "2", "3", "4"]
)

C:\Users\pk\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:41<00:00, 2.00MiB/s]


In [2]:
results = collection.query(
    query_texts=["Tell me about cricket"],
    n_results=2
)

print(results)

{'ids': [['2', '1']], 'embeddings': None, 'documents': [['Babar scored a century in cricket.', 'Pakistan won the cricket match.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None]], 'distances': [[1.0164419412612915, 1.0227837562561035]]}


In [3]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Embedding model load karo
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Documents
documents = [
    "I love playing cricket.",
    "Babar Azam is a famous cricket player.",
    "Apple stock price increased today.",
    "Microsoft released a new AI model.",
    "Pakistan won the cricket match."
]

# 3. Documents ko embeddings mein convert karo
embeddings = model.encode(documents)

# FAISS ko float32 chahiye
embeddings = np.array(embeddings).astype("float32")

print("Embedding shape:", embeddings.shape)

# 4. FAISS index create karo
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

# 5. Embeddings FAISS mein add karo
index.add(embeddings)

print("Total vectors:", index.ntotal)

# 6. User ki query
query = "Tell me something about cricket"

# 7. Query ko embedding mein convert karo
query_embedding = model.encode([query])
query_embedding = np.array(query_embedding).astype("float32")

# 8. Similarity search
k = 3

distances, indices = index.search(
    query_embedding,
    k
)

# 9. Results print karo
print("\nSearch Results:")

for i in range(k):
    idx = indices[0][i]
    distance = distances[0][i]

    print(f"\nRank: {i + 1}")
    print("Document:", documents[idx])
    print("Distance:", distance)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (5, 384)
Total vectors: 5

Search Results:

Rank: 1
Document: I love playing cricket.
Distance: 0.5703721

Rank: 2
Document: Babar Azam is a famous cricket player.
Distance: 1.1036669

Rank: 3
Document: Pakistan won the cricket match.
Distance: 1.1039745
